In [3]:
from typing import TypedDict

class Car(TypedDict):
    make: str
    model: str
    year: int
    electric: bool

In [5]:
c=Car(make="Tesla",model="Model 3",year=2020,electric=True)
c1=Car(make="Ford",model="Mustang",year="1967",electric=False) #passing the  values into str inplace of int  for year field b ut this won't create any issue as this is not validating on runtime this is making hint for developer only
print(c)
print(c1)

{'make': 'Tesla', 'model': 'Model 3', 'year': 2020, 'electric': True}
{'make': 'Ford', 'model': 'Mustang', 'year': '1967', 'electric': False}


In [9]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os
from dotenv import load_dotenv
load_dotenv()

def model():
    model =ChatHuggingFace(llm=HuggingFaceEndpoint(
        repo_id="openai/gpt-oss-20b",
        task="text-generation",
        temperature=0,
        max_new_tokens=1024,
        huggingfacehub_api_token=os.getenv("HUGGINFACE_API_KEY"))
    )
    return model
model=model()

## How TypedDict guides LLM output
When you use a `TypedDict` as a schema for structured output with an LLM, a prompt is generated behind the scenes to instruct the model. For example:
```
You are an AI assistant that extracts structured insights from a given review. Extract:
- summary: a brief summary of the review
- sentiment: overall tone of the review (positive, neutral, negative)
Return the result as JSON.
```
This ensures the model's response matches the structure defined by the `TypedDict`.

In [14]:
from typing import TypedDict,Annotated,Optional
#schema 
class review(TypedDict):
    summary: Annotated[str, "A concise summary of the review"]
    sentiment: Annotated[str, "The overall sentiment of the review, e.g., positive, negative, neutral"]

structed_model=model.with_structured_output(review)

result=structed_model.invoke("Overall, my experience has been quite good. The platform is easy to use, the features work as expected, and the overall performance is smooth. Customer support could be a bit faster, but the service itself delivers what it promises. With a few improvements, this could easily become an excellent experience. Would recommend it to others.")
print(result)
print(result['summary'])
print(result['sentiment'])  

{'sentiment': 'positive', 'summary': 'The user had a good experience overall, praised ease of use, performance, and features, but noted customer support could be faster. They recommend the service.'}
The user had a good experience overall, praised ease of use, performance, and features, but noted customer support could be faster. They recommend the service.
positive


In [19]:
# we are adding big review and we are also adding 
from typing import TypedDict,Annotated,Optional
# #schema 
class review(TypedDict):
    key_thems=Annotated[list[str],"Key themes discussed in the review"]
    summary: Annotated[str, "A concise summary of the review"]
    sentiment: Annotated[str, "The overall sentiment of the review, e.g., positive, negative, neutral"]
    pros: Annotated[Optional[list[str]], "List of pros mentioned in the review"]
    cons: Annotated[Optional[list[str]],"LIST OF CONS MENTIONS IN THE REVIEW"]
    name:Annotated[Optional[str],"name of reviewer"]

structed_model=model.with_structured_output(review)

result=structed_model.invoke("""I’ve been using the iPhone 14 for a good amount of time now, and overall, it delivers a very polished and reliable smartphone experience—exactly what people expect from Apple.

The design is familiar but refined. Apple hasn’t changed the overall look drastically, but the phone feels premium, well-balanced, and solid in hand. The glass back and aluminum frame give it a sturdy feel, and the phone doesn’t feel bulky despite its premium build. It’s comfortable to use one-handed and fits easily into daily life.

The display is one of the highlights. The Super Retina XDR display is sharp, bright, and color-accurate. Whether you’re watching videos, scrolling through social media, or reading for long periods, the screen is easy on the eyes. Outdoor visibility is excellent, and the overall viewing experience feels smooth and premium, even though the refresh rate remains standard.

In terms of performance, the iPhone 14 is extremely reliable. Everyday tasks like browsing, multitasking, camera usage, and app switching feel effortless. Apps open quickly, animations are smooth, and the phone rarely slows down. Even with heavy usage, the device remains stable and efficient, which makes it feel dependable over the long term.

The camera system performs very well, especially for everyday photography. Photos come out sharp with accurate colors and strong dynamic range. Night photography is noticeably improved, producing clearer and brighter images without excessive noise. Video recording is a major strength—stabilization, clarity, and color consistency are among the best in the smartphone market. The front camera also performs well for selfies and video calls.

Battery life is decent and consistent. The phone comfortably lasts a full day with moderate to heavy usage, including browsing, media consumption, and navigation. While it may not be the longest-lasting battery in its class, Apple’s optimization ensures predictable and stable performance throughout the day.

The software experience (iOS) is one of the iPhone 14’s strongest points. The interface is clean, intuitive, and well-optimized. iOS feels smooth and polished, with long-term software updates being a major advantage. Security, privacy features, and ecosystem integration—especially with other Apple devices like MacBooks, iPads, and AirPods—add significant value.

One notable aspect is the ecosystem experience. Features like AirDrop, iMessage, FaceTime, and seamless device syncing make daily tasks easier, especially if you’re already using Apple products. This interconnected experience is something many users appreciate over time.

However, the iPhone 14 does have some limitations. Charging speed is relatively slow compared to competitors, and the absence of major design changes may feel underwhelming for users upgrading from recent iPhone models. Additionally, the lack of expandable storage and limited customization options may not suit everyone.

Build quality and reliability are excellent. The phone feels durable, and features like water resistance and improved safety options add an extra layer of confidence for daily use.
""")
print(result)
print("pros:",result['pros'],end="\n")
print("cons:",result['cons'],end="\n")
#print("name:",result['name'],end="\n")


{'cons': ['Slow charging speed compared to competitors', 'No major design changes, may feel underwhelming for some', 'Lack of expandable storage', 'Limited customization options'], 'name': 'Unnamed Review', 'pros': ['Refined design and premium feel', 'Super Retina XDR display with excellent brightness and color accuracy', 'Reliable performance and smooth multitasking', 'Strong camera performance, especially night and video', 'Decent battery life', 'Intuitive iOS with long-term updates and privacy features', 'Strong ecosystem integration with Apple devices'], 'sentiment': 'positive', 'summary': 'The iPhone\u202f14 offers a polished, reliable experience with a premium build, sharp display, strong camera, and smooth software, but it falls short on charging speed, design novelty, expandable storage, and customization.'}
pros: ['Refined design and premium feel', 'Super Retina XDR display with excellent brightness and color accuracy', 'Reliable performance and smooth multitasking', 'Strong c

# Pydantic 
1. Pydantic is a Python library that validates and parses data using type hints.
You define what data should look like, and Pydantic makes sure reality obeys.

Think of it as:

“Type hints that actually do something at runtime.”
2. The problem Pydantic solves

In plain Python, this is legal:

```python
age = "twenty five"
```

Python shrugs. Your app breaks later.

Pydantic steps in and says:

“No. If you said age: int, then it better be an integer—or I’ll stop you immediately.”

```python
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int
    email: str
```
``` python
user = User(name="Vinod", age="25", email="a@b.com")
# works >> "25"converted to 25
User(name="Vinod", age="abc", email="a@b.com")
Error: >> invalid integer
```
One-line takeaway

TypedDict = documentation + static hints

Pydantic = enforced data contracts at runtime

If data comes from outside your code (API, user, file) → Pydantic
If data stays inside trusted code → TypedDict

In [32]:
from pydantic import BaseModel, Field
from typing import Optional

class student(BaseModel):
    name: str = Field(..., description="The full name of the student")
    age: int = Field(..., description="The age of the student in years")
    grade: str = Field(..., description="The current grade level of the student")
    gpa: float = Field(gt=0.0,lt=5, description="The student's Grade Point Average")
    email: Optional[str] = Field(None, description="The student's email address")

s=student(name="John Doe",age=20,grade="Junior",gpa="4.8")
print(s)
#s1=student(name="Jane Smith",age="Twenty",grade="Senior",gpa=3.9) #passing the age in str format instead of int but this will raise error at runtime as pydantic validate the data at runtime
s2=student(name="Jane Smith",age=21,grade="Senior",gpa=3.9,email="abs@daasai.com")
print(s2)

name='John Doe' age=20 grade='Junior' gpa=4.8 email=None
name='Jane Smith' age=21 grade='Senior' gpa=3.9 email='abs@daasai.com'


In [33]:
student_json=s2.model_dump_json()
print(student_json)

{"name":"Jane Smith","age":21,"grade":"Senior","gpa":3.9,"email":"abs@daasai.com"}


In [40]:

from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field


# schema
class Review(BaseModel):

    key_themes: list[str] = Field(description="Write down all the key themes discussed in the review in a list")
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg"] = Field(description="Return sentiment of the review either negative, positive or neutral")
    pros: Optional[list[str]] = Field(default=None, description="Write down all the pros inside a list")
    cons: Optional[list[str]] = Field(default=None, description="Write down all the cons inside a list")
    name: Optional[str] = Field(default=None, description="Write the name of the reviewer")
    

structured_model = model.with_structured_output(Review,model_return_type="pydantic")

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh
""")

print(result)

ValueError: Received unsupported arguments {'model_return_type': 'pydantic'}

In [38]:
from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field


# schema
json_schema = {
  "title": "Review",
  "type": "object",
  "properties": {
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write down all the key themes discussed in the review in a list"
    },
    "summary": {
      "type": "string",
      "description": "A brief summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["pos", "neg"],
      "description": "Return sentiment of the review either negative, positive or neutral"
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the pros inside a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the cons inside a list"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer"
    }
  },
  "required": ["key_themes", "summary", "sentiment"]
}


structured_model = model.with_structured_output(json_schema)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh
""")

print(result)

{'cons': ['Weight and size make it uncomfortable for one-handed use', 'Samsung’s One UI still has bloatware with unnecessary Samsung apps', 'High $1,300 price tag'], 'key_themes': ['Powerful Snapdragon 8 Gen 3 processor', 'High-resolution 200MP camera with night mode', 'Long battery life and fast charging', 'S-Pen integration', 'Device weight and ergonomics', 'Bloatware concerns', 'High price'], 'name': 'Nitish Singh', 'pros': ['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'], 'sentiment': 'pos', 'summary': 'Nitish Singh praises the Galaxy S24 Ultra for its powerful processor, stunning 200MP camera with excellent night mode and zoom, long battery life with fast charging, and useful S‑Pen integration, but he notes its hefty size, bloatware and high price as drawbacks.'}
